In [2]:
pip install pandas scikit-learn jupyter


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
from pathlib import Path

df = pd.read_csv(Path(r"C:\Users\bhava\Downloads\ted_talks.csv\2020-05-01\ted_talks_es.csv"))

**Load data the data set**

In [6]:
file_path = Path("C:\\Users\\bhava\\Downloads\\ted_talks.csv\\2020-05-01\\ted_talks_es.csv")

if not file_path.exists():
    file_path = Path("C:\\Users\\bhava\\Downloads\\ted_talks.csv\\2020-05-01\\ted_talks_es.csv")

if not file_path.exists():
    raise FileNotFoundError(
        "Please check the location of ted_talks_cleaned.csv"
    )

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())

df.head()

Dataset loaded successfully!
Dataset shape: (3921, 19)

Available columns:
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript']


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript
0,1,Al Gore sobre cómo evitar la crisis climática,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523396,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,Con el mismo humor y humanidad que irradió en ...,Muchas gracias Chris. Y es en verdad un gran h...
1,7,"David Pogue dice ""La Simplicidad Vende""",David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920803,2006-02-24,2006-06-27,TED2006,en,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",124.0,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,"El columnista del New York Times, David Pogue,...","Hola contestadora automática, mi vieja amiga. ..."
2,53,Un recorrido por la renovación urbana de la ma...,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664029,2006-02-26,2006-06-27,TED2006,en,"['ar', 'bg', 'bn', 'ca', 'cs', 'de', 'en', 'es...",219.0,1116,"['MacArthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"En una charla altamente emotiva, la activista ...","Si están presentes aquí hoy, y estoy muy conte..."
3,66,Ken Robinson dice que las escuelas matan la cr...,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65052534,2006-02-25,2006-06-27,TED2006,en,"['af', 'ar', 'az', 'be', 'bg', 'bn', 'ca', 'cs...",4931.0,1164,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson plantea de manera entretenida...,"Buenos días. ¿Cómo están? Ha sido increíble, ¿..."
4,92,Hans Rosling nos muestra las mejores estadísti...,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501766,2006-02-22,2006-06-27,TED2006,en,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",628.0,1190,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,Una manera única de presentar datos. Con la en...,"Hace unos 10 años, emprendí la tarea de enseña..."


**check the title and combined text columns**

In [7]:
# Find the title column
title_candidates = ["title", "talk_title", "name"]

title_col = next(
    (c for c in title_candidates if c in df.columns.str.lower()),
    None
)

# Match column names without changing the original dataset
column_lookup = {c.lower(): c for c in df.columns}

title_col = next(
    (column_lookup[c] for c in title_candidates
     if c in column_lookup),
    None
)

# Find the combined text column
text_candidates = [
    "combined_text", "cleaned_text", "text",
    "combined", "content", "transcript"
]

text_col = next(
    (column_lookup[c] for c in text_candidates
     if c in column_lookup),
    None
)

print("Title column:", title_col)
print("Text column:", text_col)

if title_col is None:
    raise ValueError("Please identify your TED Talk title column.")

if text_col is None:
    raise ValueError(
        "Please identify your combined text column from df.columns."
    )

df[[title_col, text_col]].head()

Title column: title
Text column: transcript


,title,transcript
0,Al Gore sobre cómo evitar la crisis climática,Muchas gracias Chris. Y es en verdad un gran h...
1,"David Pogue dice ""La Simplicidad Vende""","Hola contestadora automática, mi vieja amiga. ..."
2,Un recorrido por la renovación urbana de la ma...,"Si están presentes aquí hoy, y estoy muy conte..."
3,Ken Robinson dice que las escuelas matan la cr...,"Buenos días. ¿Cómo están? Ha sido increíble, ¿..."
4,Hans Rosling nos muestra las mejores estadísti...,"Hace unos 10 años, emprendí la tarea de enseña..."


**prepare usable records**

In [8]:
df[title_col] = df[title_col].fillna("").astype(str).str.strip()
df[text_col] = df[text_col].fillna("").astype(str).str.strip()

usable_df = df[
    (df[title_col] != "") &
    (df[text_col] != "")
].copy()

usable_df = usable_df.reset_index(drop=True)

print("Total records:", len(df))
print("Usable records:", len(usable_df))
print("Records excluded due to empty title or text:",
      len(df) - len(usable_df))

if len(usable_df) < 2:
    raise ValueError("At least two usable TED Talks are required.")

usable_df[[title_col, text_col]].head()

Total records: 3921
Usable records: 3921
Records excluded due to empty title or text: 0


,title,transcript
0,Al Gore sobre cómo evitar la crisis climática,Muchas gracias Chris. Y es en verdad un gran h...
1,"David Pogue dice ""La Simplicidad Vende""","Hola contestadora automática, mi vieja amiga. ..."
2,Un recorrido por la renovación urbana de la ma...,"Si están presentes aquí hoy, y estoy muy conte..."
3,Ken Robinson dice que las escuelas matan la cr...,"Buenos días. ¿Cómo están? Ha sido increíble, ¿..."
4,Hans Rosling nos muestra las mejores estadísti...,"Hace unos 10 años, emprendí la tarea de enseña..."


**Apply TF-IDF Vectorization**

In [9]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=10000,
    min_df=1
)

tfidf_matrix = vectorizer.fit_transform(usable_df[text_col])

print("TF-IDF matrix dimensions:", tfidf_matrix.shape)
print("Number of talks:", tfidf_matrix.shape[0])
print("Number of vocabulary features:", tfidf_matrix.shape[1])

vocabulary = vectorizer.get_feature_names_out()

print("\nSample vocabulary (first 30 words):")
print(vocabulary[:30])

TF-IDF matrix dimensions: (3921, 10000)
Number of talks: 3921
Number of vocabulary features: 10000

Sample vocabulary (first 30 words):
['000' '10' '100' '1000' '11' '12' '120' '125' '13' '130' '14' '140' '15'
 '150' '1500' '16' '160' '1600' '17' '18' '180' '1800' '19' '1900' '1930'
 '1945' '1950' '1960' '1968' '1970']


**calculate cosine similarity**

In [10]:
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity matrix dimensions:", similarity_matrix.shape)

print("\nSample similarity scores for the first talk:")
print(similarity_matrix[0][:10])

# A talk compared with itself has a similarity score of 1
print("\nSelf-similarity score:", similarity_matrix[0, 0])

Similarity matrix dimensions: (3921, 3921)

Sample similarity scores for the first talk:
[1.         0.56306854 0.56407265 0.59533661 0.50478628 0.4702069
 0.5650542  0.5904588  0.56536384 0.59357858]

Self-similarity score: 1.0000000000000004


**Generate recommendations for three TED talks**

In [11]:
def recommend_talks(talk_index, top_n=5):
    scores = similarity_matrix[talk_index].copy()

    # Exclude the selected talk itself
    scores[talk_index] = -1

    # Select the highest similarity scores
    top_indices = np.argsort(scores)[::-1][:top_n]

    recommendations = pd.DataFrame({
        "Recommended Talk": usable_df.iloc[top_indices][title_col].values,
        "Similarity Score": scores[top_indices]
    })

    return recommendations.reset_index(drop=True)


# Select three distinct records from the dataset
selected_indices = list(dict.fromkeys([
    0,
    len(usable_df) // 2,
    len(usable_df) - 1
]))

for idx in selected_indices:
    print("\n" + "=" * 60)
    print("SELECTED TALK:", usable_df.iloc[idx][title_col])
    print("=" * 60)

    recommendations = recommend_talks(idx, top_n=5)
    display(recommendations)


SELECTED TALK: Al Gore sobre cómo evitar la crisis climática


,Recommended Talk,Similarity Score
0,La convocatoria a la acción de Bono por África,0.647009
1,Cómo podemos cambiar el rumbo del clima,0.640860
2,Sir Ken Robinson: ¡A iniciar la revolución del...,0.625074
3,Nomi Klein: Adictos al riesgo,0.624061
4,Al Gore: Una nueva concepción de la crisis cli...,0.622122



SELECTED TALK: Así se ve el amor perdurable


,Recommended Talk,Similarity Score
0,El Personaje Norteamericano según Anna Deveare...,0.557480
1,Bruce Feiler: el consejo de papás,0.557443
2,Marc Pachter: El Arte de la Entrevista,0.551721
3,Las matemáticas del amor,0.545322
4,"Andrew Solomon: Amor, a cualquier costo",0.542706



SELECTED TALK: La oscura historia de las pruebas de coeficiente intelectual - Stefan C. Dombrowski


,Recommended Talk,Similarity Score
0,Cómo los juegos hacen más inteligentes a los n...,0.429283
1,David Brooks: el animal social,0.425629
2,¿Por qué nuestros niveles de inteligencia son ...,0.424107
3,Wade Davis visita culturas en peligro,0.418547
4,El gran alcance de la razón,0.417581


In [13]:
all_results = []

for idx in selected_indices:
    recommendations = recommend_talks(idx, top_n=5)

    for _, row in recommendations.iterrows():
        all_results.append({
            "Selected Talk": usable_df.iloc[idx][title_col],
            "Recommended Talk": row["Recommended Talk"],
            "Similarity Score": round(
                float(row["Similarity Score"]), 4
            )
        })

results_df = pd.DataFrame(all_results)

output_path = Path("C:\\Users\\bhava\\Downloads\\ted_talks.csv\\2020-05-01\\ted_talks_es.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(output_path, index=False)

print("Recommendations saved to:", output_path)
display(results_df)

Recommendations saved to: C:\Users\bhava\Downloads\ted_talks.csv\2020-05-01\ted_talks_es.csv


,Selected Talk,Recommended Talk,Similarity Score
0,Al Gore sobre cómo evitar la crisis climática,La convocatoria a la acción de Bono por África,0.6470
1,Al Gore sobre cómo evitar la crisis climática,Cómo podemos cambiar el rumbo del clima,0.6409
2,Al Gore sobre cómo evitar la crisis climática,Sir Ken Robinson: ¡A iniciar la revolución del...,0.6251
3,Al Gore sobre cómo evitar la crisis climática,Nomi Klein: Adictos al riesgo,0.6241
4,Al Gore sobre cómo evitar la crisis climática,Al Gore: Una nueva concepción de la crisis cli...,0.6221
5,Así se ve el amor perdurable,El Personaje Norteamericano según Anna Deveare...,0.5575
6,Así se ve el amor perdurable,Bruce Feiler: el consejo de papás,0.5574
7,Así se ve el amor perdurable,Marc Pachter: El Arte de la Entrevista,0.5517
8,Así se ve el amor perdurable,Las matemáticas del amor,0.5453
9,Así se ve el amor perdurable,"Andrew Solomon: Amor, a cualquier costo",0.5427


In [14]:
print("Recommendation score summary:")
print(results_df["Similarity Score"].describe())

print("\nLowest-scoring recommendations:")
display(results_df.nsmallest(5, "Similarity Score"))

print("\nHighest-scoring recommendations:")
display(results_df.nlargest(5, "Similarity Score"))

Recommendation score summary:
count    15.000000
mean      0.535260
std       0.089302
min       0.417600
25%       0.427450
50%       0.551700
75%       0.623100
max       0.647000
Name: Similarity Score, dtype: float64

Lowest-scoring recommendations:


,Selected Talk,Recommended Talk,Similarity Score
14,La oscura historia de las pruebas de coeficien...,El gran alcance de la razón,0.4176
13,La oscura historia de las pruebas de coeficien...,Wade Davis visita culturas en peligro,0.4185
12,La oscura historia de las pruebas de coeficien...,¿Por qué nuestros niveles de inteligencia son ...,0.4241
11,La oscura historia de las pruebas de coeficien...,David Brooks: el animal social,0.4256
10,La oscura historia de las pruebas de coeficien...,Cómo los juegos hacen más inteligentes a los n...,0.4293



Highest-scoring recommendations:


,Selected Talk,Recommended Talk,Similarity Score
0,Al Gore sobre cómo evitar la crisis climática,La convocatoria a la acción de Bono por África,0.6470
1,Al Gore sobre cómo evitar la crisis climática,Cómo podemos cambiar el rumbo del clima,0.6409
2,Al Gore sobre cómo evitar la crisis climática,Sir Ken Robinson: ¡A iniciar la revolución del...,0.6251
3,Al Gore sobre cómo evitar la crisis climática,Nomi Klein: Adictos al riesgo,0.6241
4,Al Gore sobre cómo evitar la crisis climática,Al Gore: Una nueva concepción de la crisis cli...,0.6221
